# 水側控制回測

回到 09/21~10/12 的每一個小時，依當時的資訊給出建議（開哪幾台、出水溫度設定），與實際運轉對照，並推估耗電差。

- **建議怎麼來**：`model/control.py` 依預測負荷呼叫 `setpoint.control_rule`；規則參數都在 `.env`
- **規則**：見 onoff_control.ipynb 第 4 節（台數）、setpoint_control.ipynb 第 6 節（出水設定）
- **建議依據**：負荷用負荷模型預測值（不用天氣資料）；流量用穩定運轉時的典型值；只在上班時段 08:00~17:00（08~16 時各整點）給建議（冰機約 07:36 開、17:24 關）
- **推估耗電**：建議運轉 = 實測 −（模型(實際) − 模型(建議)）。建議的出水設定套用到實際負荷，兩台同設定時負荷依該小時實測流量比例分配；CH-2 用灰箱模型，CH-1 用性能曲線 × 出水溫度修正（保守法）
- **評估時段**：只比較每種控制方式都有建議的上班時段（09/21~10/12）中穩定運轉的小時，以及開停機小時中穩定運轉的分段（依分鐘加權）

## 1. 環境設定與資料

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# 切到專案根目錄，才 import 得到 model、research
if Path.cwd().name == "analysis":
    os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from data.sensor import is_running, load_chiller
from model import control, recall, rules
from model.train import MODEL_ID
from research import backtest as bt
from research import ch2_model as ch2
from research import chiller
from research.ch2_model import WARMUP_MIN
from research.chart_style import date_axis, show

pio.renderers.default = "jupyterlab"

TZ = "Asia/Taipei"
START = pd.Timestamp("2026-09-17", tz=TZ)
END = pd.Timestamp("2026-10-14", tz=TZ)
SHUTDOWN_MIN = 10  # 停機前 10 分鐘為暫態，排除
MIN_STEADY = 45  # 每小時至少穩定運轉 45 分鐘才評估

In [2]:
# 模型：台數控制性能曲線、CH-2 灰箱耗電模型
curves = await chiller.load_curves()
trainer = ch2.PowerModelTrainer()
await trainer.main()
ch2_model = trainer.model
# 出水每調高 1°C，耗電減少的比例
sensitivity = -ch2.change_pct(ch2_model, trainer.df, 1.0) / 100

# CH-2 模型的資料範圍（超出為外推）
T_RANGE = (trainer.df["T_CHWS"].min(), trainer.df["T_CHWS"].max())
Q_RANGE = (ch2.cooling_rt(trainer.df).min(), ch2.cooling_rt(trainer.df).max())
print(
    f"CH-2 灰箱模型：CV(RMSE) {trainer.metric['CV(RMSE) %']:.1f}%，"
    f"出水每調高 1°C 耗電約 −{sensitivity * 100:.2f}%；"
    f"建模範圍 出水 {T_RANGE[0]:.1f}~{T_RANGE[1]:.1f}°C、"
    f"負荷 {Q_RANGE[0]:.0f}~{Q_RANGE[1]:.0f} RT"
)
print(
    f".env 規則參數：出水下限 {rules.T_MIN}°C、上限 {rules.T_MAX}°C、"
    f"回水上限 {rules.T_RETURN}°C、並聯門檻 {rules.PARALLEL_ABOVE} RT"
)

CH-2 灰箱模型：CV(RMSE) 3.1%，出水每調高 1°C 耗電約 −1.64%；建模範圍 出水 8.5~9.8°C、負荷 218~361 RT
.env 規則參數：出水下限 8.7°C、上限 12.0°C、回水上限 13.9°C、並聯門檻 380 RT


In [3]:
# 穩定運轉的分鐘：排除開機後 WARMUP_MIN 分鐘、停機前 SHUTDOWN_MIN 分鐘
def steady_minutes(minutes: pd.DataFrame) -> pd.Series:
    running = is_running(minutes)
    segment = (running != running.shift()).cumsum()  # 每次開機或停機就換一段
    run_min = running.groupby(segment).cumsum()  # 已運轉幾分鐘
    left_min = running.groupby(segment).transform("sum") - run_min  # 離停機還有幾分鐘
    return (
        running
        & (run_min > WARMUP_MIN)
        & (left_min >= SHUTDOWN_MIN)
        & (minutes["Q_COOL"] > 50)
    )


# 逐時資料：只取穩定運轉分鐘的平均；欄位尾碼 1 = CH-1、2 = CH-2
parts = []
minute_data = {}
for number, chiller_config in enumerate(rules.CHILLERS.values(), start=1):
    minutes = (await load_chiller(chiller_config["id"])).resample("1min").mean()
    steady = steady_minutes(minutes)
    # 每分鐘資料（開停機的小時分段用）
    minute_data[number] = minutes.assign(STEADY=steady, ON=is_running(minutes))
    steady_avg = minutes[steady].resample("1h").mean()
    parts.append(
        pd.DataFrame(
            {
                f"Q{number}": steady_avg["Q_COOL"],
                f"T{number}": steady_avg["T_CHWS"],
                f"R{number}": steady_avg["T_CHWR"],
                f"F{number}": steady_avg["FLOW"],
                f"KW{number}": steady_avg["kW"],
                # 穩定運轉滿 MIN_STEADY 分鐘
                f"ON{number}": steady.resample("1h").sum() >= MIN_STEADY,
                # 整小時停機
                f"OFF{number}": is_running(minutes).resample("1h").sum() == 0,
            }
        )
    )
    # 時序圖用（含開停機的小時）：負荷、出水取運轉分鐘平均；整小時負荷、耗電停機的分鐘記 0
    running = is_running(minutes)
    running_avg = minutes[running].resample("1h").mean()
    hour_avg = minutes[["Q_COOL", "kW"]].where(running, 0).resample("1h").mean()
    parts.append(
        pd.DataFrame(
            {
                f"QR{number}": running_avg["Q_COOL"],
                f"TR{number}": running_avg["T_CHWS"],
                f"QH{number}": hour_avg["Q_COOL"],
                f"KWH{number}": hour_avg["kW"],
                f"RUN{number}": running.resample("1h").sum() > 0,
            }
        )
    )
hour_range = pd.date_range(START, END, freq="1h", inclusive="left")
hourly = pd.concat(parts, axis=1).reindex(hour_range)
for number in (1, 2):  # 沒有資料的小時：不算穩定運轉、算停機
    hourly[f"ON{number}"] = hourly[f"ON{number}"].eq(True)
    hourly[f"OFF{number}"] = hourly[f"OFF{number}"].ne(False)
    hourly[f"RUN{number}"] = hourly[f"RUN{number}"].eq(True)

# 負荷模型預測（同 onoff_control.ipynb）；index 加 1 小時才是被預測的小時
recaller = recall.ModelRecaller(MODEL_ID, int(START.timestamp()), int(END.timestamp()))
await recaller.recall_model()
prediction_frame, _ = await recaller.predict()
load_prediction = prediction_frame["Q_COOL_PRED"]
predicted_hours = load_prediction.index + pd.Timedelta(hours=1)
hourly["LOAD_PRED"] = load_prediction.set_axis(predicted_hours).reindex(hourly.index)

# 只評估「兩台並聯」與「僅 CH-2」穩定運轉的小時
evaluable = (hourly.ON1 & hourly.ON2) | (hourly.ON2 & hourly.OFF1)
hourly["ON1"] &= evaluable
hourly["ON2"] &= evaluable
# 實測耗電
ch1_meter = hourly["KW1"].where(hourly.ON1, 0)
ch2_meter = hourly["KW2"].where(hourly.ON2, 0)
hourly["METER"] = ch1_meter + ch2_meter
parallel_count = (hourly.ON1 & hourly.ON2).sum()
ch2_only_count = (hourly.ON2 & ~hourly.ON1).sum()
print(
    f"可評估 {evaluable.sum()} 小時"
    f"（並聯 {parallel_count}、僅 CH-2 {ch2_only_count}）"
)

# 開停機的小時（如 CH-1 中途停機）：拆成「兩台穩定並聯」與「只開 CH-2 穩定、CH-1 停機」兩段各自評估，
# 兩段分鐘合計 ≥ MIN_STEADY 才補；其餘分鐘（開停機暫態）照實測
minute_index = pd.date_range(START, END, freq="1min", inclusive="left")
minute_frames = {}
for number in (1, 2):
    minute_frames[number] = minute_data[number].reindex(minute_index)
ch1_steady = minute_frames[1]["STEADY"].eq(True)
ch2_steady = minute_frames[2]["STEADY"].eq(True)
ch1_on = minute_frames[1]["ON"].eq(True)
# 是否並聯：該模式的分鐘
mode_minutes = {True: ch1_steady & ch2_steady, False: ch2_steady & ~ch1_on}
good_minutes = sum(mask.resample("1h").sum() for mask in mode_minutes.values())
good_minutes = good_minutes.reindex(hourly.index).fillna(0)
split_hours = ~evaluable & (good_minutes >= MIN_STEADY)
segment_rows = []
for hour in hourly.index[split_hours]:
    for is_parallel, mask in mode_minutes.items():
        selected = mask.loc[hour : hour + pd.Timedelta(minutes=59)]
        if not selected.any():
            continue
        selected_index = selected.index[selected]
        ch1_avg = minute_frames[1].loc[selected_index].mean()
        ch2_avg = minute_frames[2].loc[selected_index].mean()
        # 只開 CH-2 時，CH-1 負荷、流量、耗電記 0，出水留空
        if is_parallel:
            ch1_load = ch1_avg["Q_COOL"]
            ch1_supply = ch1_avg["T_CHWS"]
            ch1_flow = ch1_avg["FLOW"]
            ch1_kw = ch1_avg["kW"]
        else:
            ch1_load = 0.0
            ch1_supply = np.nan
            ch1_flow = 0.0
            ch1_kw = 0.0
        segment_rows.append(
            {
                "HOUR": hour,
                "W": selected.sum() / 60,
                "ON1": is_parallel,
                "ON2": True,
                "LOAD_PRED": hourly.at[hour, "LOAD_PRED"],
                "Q1": ch1_load,
                "T1": ch1_supply,
                "F1": ch1_flow,
                "KW1": ch1_kw,
                "Q2": ch2_avg["Q_COOL"],
                "T2": ch2_avg["T_CHWS"],
                "F2": ch2_avg["FLOW"],
                "KW2": ch2_avg["kW"],
            }
        )
segments = pd.DataFrame(segment_rows)
segments["METER"] = segments.KW1 + segments.KW2
split_hour_texts = "、".join(
    f"{hour:%m/%d %H:00}" for hour in hourly.index[split_hours]
)
print(f"開停機的小時分段評估 {split_hours.sum()} 小時：" + split_hour_texts)

# 規則用的流量典型值（同 control.typical_flows：穩定運轉小時的中位數）：只開 CH-2 用 CH-2 流量、並聯用兩台合計
FLOWS = {
    "single": float(hourly.F2[hourly.ON2 & ~hourly.ON1].median()),
    "parallel": float((hourly.F1 + hourly.F2)[hourly.ON1 & hourly.ON2].median()),
}
print(
    f"流量典型值：只開 CH-2 {FLOWS['single']:,.0f} LPM、"
    f"兩台並聯合計 {FLOWS['parallel']:,.0f} LPM"
)

# 實際運轉（整小時平均，同負荷模型的目標）
full_hour = {}
for number in (1, 2):
    full_hour[number] = (
        minute_data[number][["Q_COOL", "kW"]]
        .resample("1h")
        .mean()
        .reindex(hourly.index)
    )
any_running = hourly.RUN1 | hourly.RUN2
hourly["Q_ACT"] = (full_hour[1].Q_COOL.fillna(0) + full_hour[2].Q_COOL.fillna(0)).where(
    any_running
)
hourly["KW_ACT"] = (full_hour[1].kW.fillna(0) + full_hour[2].kW.fillna(0)).where(
    any_running
)
hourly["PLAN"] = np.select(
    [hourly.RUN1 & hourly.RUN2, hourly.RUN2, hourly.RUN1],
    [rules.BOTH, rules.CH2_ONLY, rules.CH1_ONLY],
    rules.OFF,
)
running = hourly["PLAN"] != rules.OFF
work_hours = (hourly.index.hour >= control.WORK_START) & (
    hourly.index.hour <= control.WORK_END
)

# 運轉方式改變時用的典型值（並聯時中位數）
parallel_hours = hourly[hourly.ON1 & hourly.ON2]
TYP = {
    "F1": parallel_hours.F1.median(),
    "F2": parallel_hours.F2.median(),
    "T1": parallel_hours.T1.median(),
    "F2_single": FLOWS["single"],
}

# 評估單位：穩定運轉的小時（權重 1）＋開停機小時的分段（權重 = 分鐘 ÷ 60）
steady_units = hourly.loc[evaluable, bt.UNIT_COLS].assign(
    HOUR=hourly.index[evaluable],
    W=1.0,
)
segment_units = segments[bt.UNIT_COLS + ["HOUR", "W"]]
units = pd.concat([steady_units, segment_units], ignore_index=True)
units["ON1"] = units["ON1"].astype(bool)
meter_kwh = (units.METER * units.W).sum()
print(
    f"評估單位 {len(units)} 個（{evaluable.sum()} 小時＋分段 {len(segments)} 段），"
    f"實測耗電 {meter_kwh:,.0f} kWh；"
    f"並聯時典型值：CH-1 {TYP['F1']:,.0f} LPM、出水 {TYP['T1']:.1f}°C，"
    f"CH-2 {TYP['F2']:,.0f} LPM"
)

可評估 103 小時（並聯 20、僅 CH-2 83）
開停機的小時分段評估 5 小時：09/29 15:00、09/30 15:00、10/01 15:00、10/02 13:00、10/05 13:00
流量典型值：只開 CH-2 3,852 LPM、兩台並聯合計 6,088 LPM
評估單位 113 個（103 小時＋分段 10 段），實測耗電 22,605 kWh；並聯時典型值：CH-1 2,781 LPM、出水 11.0°C，CH-2 3,307 LPM


In [4]:
# 時序圖用：只排有資料的小時，不同天之間插一格空白（線不跨天相連）；回傳 (橫軸標籤, 資料)
def by_day(df: pd.DataFrame) -> tuple[list, pd.DataFrame]:
    parts = []
    labels = []
    day_groups = df.groupby(df.index.strftime("%m/%d"), sort=False)
    for i, (day, group) in enumerate(day_groups):
        if i > 0:
            parts.append(pd.DataFrame(np.nan, index=[0], columns=df.columns))
            labels.append(f"｜{day}")
        parts.append(group.reset_index(drop=True))
        labels += list(group.index.strftime("%m/%d %H:00"))
    return labels, pd.concat(parts, ignore_index=True)


# 類別橫軸刻度標每天第一個小時的日期；相鄰刻度太近（運轉時數少的日子）就略過，最多約 n_max 個
def hour_axis(fig: go.Figure, labels: list, n_max: int = 12) -> None:
    # 每天第一個小時的位置（空白格之後的第一格）
    day_starts = []
    for i, tick_label in enumerate(labels):
        if tick_label.startswith("｜"):
            continue
        if i == 0 or labels[i - 1].startswith("｜"):
            day_starts.append(i)
    keep = []
    last = -np.inf
    for i in day_starts:
        if i - last >= len(labels) / n_max:
            keep.append(i)
            last = i
    fig.update_xaxes(
        type="category",
        tickvals=[labels[i] for i in keep],
        ticktext=[labels[i][:5] for i in keep],
        tickangle=0,
    )

## 2. 負荷預測與台數建議

- 建議依 `control.recommend`（同業主 API）：上班時段 08:00~17:00 逐時給建議，台數照 onoff_control.ipynb 第 4 節、出水照 setpoint_control.ipynb 第 6 節
- 第 3、4 節的台數與出水都用這組建議
- **比較時段**：只比較逐時、3 段、2 段、1 段**都有建議**的小時（資料剛開始、預測不完整的時段不比較）；第 3~6 節與各圖都只用這些時段

In [5]:
# 各種控制方式的逐時建議（上班 08:00~17:00）；逐時為第 3~5 節的建議，第 6 節比較各段數
MODES = {None: "逐時", **control.SEGMENT_NAMES}
recommendations = {}
for segment_count in MODES:
    recommendations[segment_count] = control.recommend(
        hourly.LOAD_PRED, FLOWS, segment_count
    )
recommendation = recommendations[None]

# 比較時段：每種方式都有建議的小時；評估單位只留比較時段
has_recommendation = [
    mode_recommendation["建議方案"].notna()
    for mode_recommendation in recommendations.values()
]
comparable_hour = pd.concat(has_recommendation, axis=1).all(axis=1)
comparable = comparable_hour & running
units = units[units.HOUR.isin(hourly.index[comparable_hour])].reset_index(drop=True)
meter_kwh = (units.METER * units.W).sum()
N_HR = int((units.W == 1).sum())
N_SEG = int((units.W < 1).sum())
days = hourly.index[comparable].normalize().unique()
print(
    f"比較時段 {days.min():%m/%d}~{days.max():%m/%d}"
    f" 上班時段開機 {comparable.sum()} 小時；"
    f"評估 {N_HR} 小時＋開停機分段 {N_SEG} 段，實測耗電 {meter_kwh:,.0f} kWh"
)

# 每日 × 每小時：建議（逐時）與實際運轉方式
CATS = {  # 名稱：(建議, 實際, 顏色)；一致淺色、不同深色；只開 CH-2 為橘、並聯為藍（依建議）
    "一致：只開 CH-2": (rules.CH2_ONLY, rules.CH2_ONLY, "#f5a582"),
    "一致：兩台並聯": (rules.BOTH, rules.BOTH, "#86b6ef"),
    "不同：建議並聯（實際只開 CH-2）": (rules.BOTH, rules.CH2_ONLY, "#1c5cab"),
    "不同：建議只開 CH-2（實際並聯）": (rules.CH2_ONLY, rules.BOTH, "#a33f17"),
}
compare_hours = hourly[comparable].assign(
    REC=recommendation.loc[comparable, "建議方案"]
)
y_order = [f"{day:%m/%d}（{'一二三四五六日'[day.weekday()]}）" for day in days]
fig = go.Figure()
for category_name, (recommended_plan, actual_plan, color) in CATS.items():
    category_hours = compare_hours[
        (compare_hours.REC == recommended_plan) & (compare_hours.PLAN == actual_plan)
    ]
    if len(category_hours):
        hover_data = np.stack(
            [category_hours.LOAD_PRED, category_hours.Q_ACT],
            axis=-1,
        )
    else:
        hover_data = None
    fig.add_trace(
        go.Scatter(
            x=category_hours.index.strftime("%H:00"),
            y=[
                y_order[days.get_loc(hour.normalize())] for hour in category_hours.index
            ],
            mode="markers",
            name=f"{category_name}（{len(category_hours)} 小時）",
            marker=dict(symbol="square", size=26, color=color),
            customdata=hover_data,
            hovertemplate=(
                f"%{{y}} %{{x}}｜{category_name}<br>"
                "預測 %{customdata[0]:.0f} RT"
                "、實際 %{customdata[1]:.0f} RT<extra></extra>"
            ),
        )
    )
work_hour_labels = [
    f"{hour:02d}:00" for hour in range(control.WORK_START, control.WORK_END + 1)
]
fig.update_xaxes(
    categoryorder="array",
    categoryarray=work_hour_labels,
    showgrid=False,
)
fig.update_yaxes(categoryorder="array", categoryarray=y_order[::-1], showgrid=False)
fig.update_layout(margin=dict(l=110))
# 建議與實際一致的小時數
same_count = 0
for category_name, (recommended_plan, actual_plan, _) in CATS.items():
    if category_name.startswith("一致"):
        same_count += (
            (compare_hours.REC == recommended_plan)
            & (compare_hours.PLAN == actual_plan)
        ).sum()
show(
    fig,
    "建議與實際運轉方式",
    f"比較時段上班時段開機 {comparable.sum()} 小時，建議與實際一致 {same_count} 小時；"
    f"滑鼠移到格子可看預測與實際負荷",
    legend="bottom",
    height=60 + 34 * len(days) + 150,
)

比較時段 09/21~10/12 上班時段開機 117 小時；評估 92 小時＋開停機分段 10 段，實測耗電 20,277 kWh


## 3. 台數控制回測

- **台數**照第 2 節建議，**出水維持現況**（CH-2 用實測；建議並聯但 CH-1 沒開時，CH-1 出水、兩台流量用並聯時中位數）
- **建議運轉耗電** = 實測 −（模型(現況) − 模型(台數照建議)）；CH-2 用灰箱模型，CH-1 用性能曲線 × 出水溫度修正
- 建議的台數做不到（如超出 CH-2 單機範圍）時沿用現況

In [6]:
result = bt.evaluate(units, recommendation, curves, ch2_model, sensitivity, TYP)
# 各步驟相對前一步
step_saving = {
    step: bt.saving(result, units, step, previous)
    for step, previous in [("S1", "S0"), ("S2", "S1"), ("S3", "S2")]
}
unknown = result["S0 kW"].isna()  # 現況無法評估（以實測表示）

N_UNITS = {
    rules.OFF: 0,
    rules.TOO_LOW: 0,
    rules.CH2_ONLY: 1,
    rules.CH1_ONLY: 1,
    rules.BOTH: 2,
}
recommended_plan = recommendation["建議方案"].where(comparable)  # 無建議的小時照實際
step1_by_hour = bt.by_hour(step_saving["S1"], units).reindex(hourly.index).fillna(0)
kw_step1 = (hourly.KW_ACT - step1_by_hour).where(comparable)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("總耗電 (kW)", "運轉台數"),
)
plan_lines = [
    (
        "實際運轉",
        hourly.KW_ACT.where(comparable),
        hourly.PLAN.where(comparable),
        "#f5a582",
        2,
        "actual",
    ),
    ("台數照建議", kw_step1, recommended_plan, "#a33f17", 1.5, "plan"),
]
for line_name, power, plan, color, width, group in plan_lines:
    # 台數圖的實際運轉線加粗一點
    if group == "actual":
        plan_width = width + 1
    else:
        plan_width = width
    fig.add_trace(
        go.Scatter(
            x=power.index,
            y=power,
            mode="lines",
            name=line_name,
            legendgroup=group,
            line=dict(color=color, width=width),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=plan.index,
            y=plan.map(N_UNITS),
            mode="lines",
            name=line_name,
            legendgroup=group,
            showlegend=False,
            line=dict(color=color, width=plan_width, shape="hv"),
        ),
        row=2,
        col=1,
    )
fig.update_yaxes(tickvals=[0, 1, 2], range=[-0.2, 2.3], row=2, col=1)
fig.update_layout(hovermode="x unified")
date_axis(fig)
fig.update_xaxes(
    range=[days.min() - pd.Timedelta(hours=12), days.max() + pd.Timedelta(hours=36)]
)
show(
    fig,
    "台數控制回測：總耗電與運轉台數",
    "比較時段；台數照建議、出水維持現況；"
    "建議運轉耗電 = 實測 −（模型(現況) − 模型(台數照建議)）",
    legend="bottom",
    height=850,
)

actual_plan = np.where(units.ON1, rules.BOTH, rules.CH2_ONLY)
changed = (result["S1 kW"] != result["S0 kW"]) & ~unknown
print(
    f"評估 {len(units)} 個單位：台數照建議節電 {step_saving['S1'].sum():,.0f} kWh"
    f"（{step_saving['S1'].sum() / meter_kwh * 100:.1f}%）；"
    f"台數改變 {int(changed.sum())} 個單位；"
    f"現況無法評估、以實測表示 {int(unknown.sum())} 個"
)
for plan_name in [rules.BOTH, rules.CH2_ONLY]:
    plan_units = actual_plan == plan_name
    print(
        f"  實際{plan_name}：{int(plan_units.sum())} 個單位，"
        f"台數照建議節電 {step_saving['S1'][plan_units].sum():,.0f} kWh"
    )

評估 102 個單位：台數照建議節電 -528 kWh（-2.6%）；台數改變 14 個單位；現況無法評估、以實測表示 0 個
  實際CH-1 + CH-2：25 個單位，台數照建議節電 9 kWh
  實際CH-2：77 個單位，台數照建議節電 -537 kWh


## 4. 出水溫度控制回測

- **台數照第 3 節**（建議），再依序套用出水建議：先改**單機出水**（建議只開 CH-2 的單位），再改**並聯出水**（建議並聯的單位，兩台同一設定）
- 兩台同設定時負荷依流量比例分配（流量用該單位實測；運轉方式改變時用中位數）
- 出水建議做不到時沿用前一步

只開 CH-2 的各小時，比較現況與建議設定的出水溫度與 CH-2 耗電。

In [7]:
# 建議與實際都只開 CH-2 的穩定運轉小時：現況與建議出水、CH-2 耗電
whole_hour = units.W == 1
ch2_only_index = units.index[
    whole_hour & ~units.ON1 & result["S2 CH-1 RT"].eq(0) & ~unknown
]
ch2_only_units = units.loc[ch2_only_index]
suggested_supply = result.loc[ch2_only_index, "S2 CH-2 °C"]
# 實測 −（模型(台數照建議) − 模型(單機出水照建議)）
suggested_kw = ch2_only_units.METER - (
    result.loc[ch2_only_index, "S1 kW"] - result.loc[ch2_only_index, "S2 kW"]
)
hour_text = ch2_only_units.HOUR.dt.strftime("%m/%d %H:00")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("CH-2 出水溫度 (°C)", "CH-2 耗電 (kW)"),
)
panels = [
    (1, ch2_only_units.T2, suggested_supply, "°C", ".1f"),
    (2, ch2_only_units.METER, suggested_kw, "kW", ".0f"),
]
for row, current_values, suggested_values, unit, value_format in panels:
    series_list = [
        (current_values, "現況", "#f5a582"),
        (suggested_values, "建議設定", "#a33f17"),
    ]
    for y_values, series_name, color in series_list:
        fig.add_trace(
            go.Scatter(
                x=ch2_only_units.Q2,
                y=y_values,
                mode="markers",
                name=series_name,
                legendgroup=series_name,
                showlegend=row == 1,
                marker=dict(size=7, color=color, opacity=0.85),
                text=hour_text,
                hovertemplate=(
                    f"%{{text}}｜負荷 %{{x:.0f}} RT｜{series_name} "
                    f"%{{y:{value_format}}} {unit}<extra></extra>"
                ),
            ),
            row=row,
            col=1,
        )
fig.update_xaxes(title_text="CH-2 負荷 (RT)", row=2, col=1)
show(
    fig,
    "只開 CH-2 時：現況與建議設定的出水溫度、耗電",
    f"建議與實際都只開 CH-2 的穩定運轉 {len(ch2_only_units)} 小時；"
    f"建議設定耗電 = 實測 −（模型現況 − 模型建議）",
    legend="bottom",
    height=800,
)
supply_change = suggested_supply - ch2_only_units.T2
print(
    f"建議比現況調高 {(supply_change > 0.05).sum()} 小時、"
    f"調低 {(supply_change < -0.05).sum()} 小時；"
    f"耗電減少 {(suggested_kw < ch2_only_units.METER).sum()} 小時、"
    f"增加 {(suggested_kw > ch2_only_units.METER).sum()} 小時；"
    f"合計 {(ch2_only_units.METER - suggested_kw).sum():,.0f} kWh"
)

建議比現況調高 60 小時、調低 2 小時；耗電減少 60 小時、增加 3 小時；合計 218 kWh


各小時的負荷，以及兩台出水設定的現況與建議（灰底為並聯）；並聯時兩台建議為同一設定。

In [8]:
# 時序圖：比較時段（含開停機的小時）；建議出水依第 2 節建議
compare_hourly = hourly[comparable]
compare_recommendation = recommendation.loc[compare_hourly.index]
compare_plan = compare_recommendation["建議方案"]
compare_setpoint = compare_recommendation["出水設定 °C"]
tick_labels, plot_data = by_day(
    pd.DataFrame(
        {
            "load": compare_hourly.Q_ACT,
            "pred": compare_hourly.LOAD_PRED,
            "T1": compare_hourly.TR1,
            "T2": compare_hourly.TR2,
            "SA": compare_setpoint.where(compare_plan == rules.CH2_ONLY),  # 單台 CH-2
            # 並聯（兩台同設定）
            "SP": compare_setpoint.where(compare_plan == rules.BOTH),
            # 灰底：兩台都有開
            "par": (compare_hourly.RUN1 & compare_hourly.RUN2).astype(float),
        }
    )
)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("總負荷 (RT)", "各機出水設定 (°C)"),
)
# (子圖列, 欄位, 名稱, 圖例, 線條, 單位)
line_specs = [
    (1, "load", "實際", "legend", dict(color="#86b6ef", width=3), "RT"),
    (1, "pred", "預測", "legend", dict(color="#1c5cab", width=1.5), "RT"),
    (
        2,
        "T1",
        "CH-1 現況",
        "legend2",
        dict(color="#86b6ef", width=2, dash="dash"),
        "°C",
    ),
    (
        2,
        "T2",
        "CH-2 現況",
        "legend2",
        dict(color="#f5a582", width=2, dash="dash"),
        "°C",
    ),
    (2, "SA", "單台 CH-2 建議", "legend2", dict(color="#a33f17", width=2), "°C"),
    (
        2,
        "SP",
        "並聯建議（兩台同設定）",
        "legend2",
        dict(color="#52514e", width=2),
        "°C",
    ),
]
for row, column, line_name, legend, line, unit in line_specs:
    if unit == "RT":
        value_format = ".0f"
    else:
        value_format = ".1f"
    fig.add_trace(
        go.Scatter(
            x=tick_labels,
            y=plot_data[column],
            mode="lines",
            name=line_name,
            legend=legend,
            line=line,
            hovertemplate=f"{line_name} %{{y:{value_format}}} {unit}<extra></extra>",
        ),
        row=row,
        col=1,
    )

# 灰底標示並聯的小時（類別軸以位置計）
parallel_flags = np.r_[0, plot_data["par"].fillna(0).astype(int), 0]
edges = np.flatnonzero(np.diff(parallel_flags))
for shade_start, shade_end in zip(edges[::2], edges[1::2]):
    fig.add_vrect(
        x0=shade_start - 0.5,
        x1=shade_end - 0.5,
        fillcolor="#e8e6e1",
        opacity=0.6,
        line_width=0,
        layer="below",
        row="all",
        col=1,
    )
fig.update_yaxes(range=[6.5, 12.5], row=2, col=1)
hour_axis(fig, tick_labels)
fig.update_layout(hovermode="x unified")
show(
    fig,
    "出水溫度設定：逐時",
    "比較時段；建議出水依第 2 節建議；灰底為兩台都有開",
    legend_groups=["負荷", "各機出水設定"],
    height=800,
)

for plan_name in [rules.CH2_ONLY, rules.BOTH]:
    plan_hours = comparable & (recommendation["建議方案"] == plan_name)
    plan_setpoints = recommendation.loc[plan_hours, "出水設定 °C"]
    print(
        f"比較時段建議 {plan_name}：{len(plan_setpoints)} 小時，"
        f"出水 {plan_setpoints.min():.1f}~{plan_setpoints.max():.1f}°C"
    )
ch2_out = (result["S3 CH-2 °C"] < T_RANGE[0] - 0.05) | (
    result["S3 CH-2 °C"] > T_RANGE[1] + 0.05
)
print(f"CH-2 建議出水超出灰箱模型範圍 {int(ch2_out.sum())} 個評估單位")

比較時段建議 CH-2：73 小時，出水 8.9~11.5°C
比較時段建議 CH-1 + CH-2：44 小時，出水 9.6~10.8°C
CH-2 建議出水超出灰箱模型範圍 72 個評估單位


In [9]:
# 回測時間序列：照建議（台數＋出水）前後的總耗電；比較時段，照建議只畫可評估的小時
total_saving = step_saving["S1"] + step_saving["S2"] + step_saving["S3"]
# 實測 −（模型現況 − 模型照建議）
kw_after = (
    hourly.KW_ACT - bt.by_hour(total_saving, units).reindex(hourly.index)
).where(comparable)
tick_labels, plot_data = by_day(
    pd.DataFrame(
        {
            "meter": hourly.KW_ACT[comparable],
            "after": kw_after[comparable],
        }
    )
)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=tick_labels,
        y=plot_data["meter"],
        mode="lines",
        name="現況",
        line=dict(color="#f5a582", width=2.5),
        hovertemplate="現況 %{y:.0f} kW<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=tick_labels,
        y=plot_data["after"],
        mode="lines+markers",
        name="照建議",
        line=dict(color="#a33f17", width=1.5),
        marker=dict(size=4),  # 前後小時無法評估時，單一小時以點呈現
        hovertemplate="照建議 %{y:.0f} kW<extra></extra>",
    )
)
fig.update_yaxes(title_text="總耗電 (kW)")
hour_axis(fig, tick_labels)
fig.update_layout(hovermode="x unified")
show(
    fig,
    "水側控制回測：總耗電",
    f"比較時段；照建議 = 實測 −（模型現況 − 模型照建議），"
    f"可評估不足 {MIN_STEADY} 分鐘的開停機小時留空",
    legend="bottom",
    height=550,
)

## 5. 節電彙總

各步驟**逐步累加**：每一步是相對前一步多省的電，合計 = 照建議操作（台數＋出水）的總節電

In [10]:
# 瀑布圖：每一段是相對前一步多省的電，累加到合計
LABELS = {"S1": "台數", "S2": "單機出水", "S3": "並聯出水"}
step_kwh = pd.Series({step: step_saving[step].sum() for step in LABELS})
step_pct = step_kwh / meter_kwh * 100
bar_texts = [
    f"{kwh:+,.0f} kWh<br>{percent:+.1f}%" for kwh, percent in zip(step_kwh, step_pct)
]
bar_texts += [f"{step_kwh.sum():,.0f} kWh<br>{step_pct.sum():.1f}%"]
fig = go.Figure(
    go.Waterfall(
        x=list(LABELS.values()) + ["合計"],
        y=list(step_kwh) + [step_kwh.sum()],
        measure=["relative"] * len(step_kwh) + ["total"],
        text=bar_texts,
        textposition="outside",
        showlegend=False,
        increasing=dict(marker=dict(color="#f5a582")),
        decreasing=dict(marker=dict(color="#86b6ef")),
        totals=dict(marker=dict(color="#a33f17")),
        connector=dict(line=dict(color="#8a8984", width=1, dash="dot")),
        hovertemplate="%{x}｜%{y:+,.0f} kWh<extra></extra>",
    )
)
cumulative = step_kwh.cumsum()
fig.update_yaxes(
    title_text="節電量 (kWh)",
    range=[min(0, cumulative.min()) * 1.3 - 50, max(cumulative.max(), 0) * 1.25],
)
fig.update_xaxes(tickangle=0, tickfont=dict(size=12))
show(
    fig,
    "照建議操作的節電（逐步累加）",
    f"比較時段 {N_HR} 小時＋開停機分段 {N_SEG} 段；淺橘為多省、淺藍為多耗；"
    f"百分比相對同期實測 {meter_kwh:,.0f} kWh",
    height=650,
)

# 依實際運轉方式拆分
actual_plan = np.where(units.ON1, rules.BOTH, rules.CH2_ONLY)
summary = pd.DataFrame(
    {
        LABELS[step]: pd.Series(step_saving[step]).groupby(actual_plan).sum()
        for step in LABELS
    }
)
summary["合計"] = summary.sum(axis=1)
summary.loc["合計"] = summary.sum()
summary.index.name = "實際運轉（kWh）"
summary.round(0)

,台數,單機出水,並聯出水,合計
實際運轉（kWh）,,,,
CH-1 + CH-2,9.0,0.0,769.0,779.0
CH-2,-537.0,217.0,258.0,-62.0
合計,-528.0,218.0,1028.0,717.0


## 6. 一天分幾段控制

業主可選一天分幾段控制：
- **3 段**（每 3 小時：08~11、11~14、14~17）
- **2 段**（上午 4 小時 08~12、下午 5 小時 12~17）
- **1 段**（整天 9 小時 08~17）

- **台數與出水**：都依段內預測負荷的第 75 百分位（`.env` 的 `CONTROL_LOAD_PCT`）；百分位 > 380 RT 整段並聯；段內有任何一小時沒有預測就不給建議
- **尖峰規則**：段內各小時超過 380 RT 的部分加總超過 95 RT（380 × 25%，`.env` 的 `CONTROL_PEAK_EXCESS`）時，整段改依最高那小時決定、兩台並聯（見第 7 節）
- **回水超過**：實際負荷下，回水比回水上限（`.env` 的 `CONTROL_T_RETURN`，13.9°C）高多少；設定越高、越常超過，末端供冷除濕越吃緊
- **為何會超過**：規則以預測負荷和流量典型值估算回水並設在 13.9°C；實際流量低於典型值、或實際負荷高於預測時，實際回水就會超過。逐時建議下，超過的時段平均高約 0.27°C，主因是實際流量低於典型值（低負荷時流量較小、並聯時合計流量較少），其次是負荷預測偏低
- 逐時為對照，業主不能選

In [11]:
LABELS6 = {key: value.replace("（", "<br>（") for key, value in MODES.items()}
block_rows = []
for segment_count, segment_label in LABELS6.items():
    segment_recommendation = recommendations[segment_count]
    segment_result = bt.evaluate(
        units, segment_recommendation, curves, ch2_model, sensitivity, TYP
    )
    saving_kwh = sum(
        bt.saving(segment_result, units, step, previous)
        for step, previous in [("S1", "S0"), ("S2", "S1"), ("S3", "S2")]
    )
    # 各小時回水超過允許水準最多幾度
    hourly_excess = bt.return_excess(segment_result).groupby(units.HOUR).max()
    parallel_units = segment_result["S3 CH-1 RT"] > 0
    block_rows.append(
        {
            "控制方式": segment_label,
            "推估節電 kWh": saving_kwh.sum(),
            "節電 %": saving_kwh.sum() / meter_kwh * 100,
            "回水超過（小時）": int((hourly_excess > 0.05).sum()),
            "回水最多超過 °C": hourly_excess.max(),
            "建議並聯（評估單位）": int(parallel_units.sum()),
            "其中實際只開 CH-2": int((parallel_units & ~units.ON1).sum()),
            "並聯單位節電 kWh": saving_kwh[units.ON1].sum(),
            "僅 CH-2 單位節電 kWh": saving_kwh[~units.ON1].sum(),
        }
    )
blocks = pd.DataFrame(block_rows).set_index("控制方式")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=("推估節電 (kWh)", "回水超過回水上限的小時"),
)
bar_colors = ["#b5b3ad"] + ["#a33f17"] * (len(blocks) - 1)  # 逐時（對照）用灰
saving_texts = [
    f"{kwh:,.0f}<br>{percent:.1f}%"
    for kwh, percent in zip(blocks["推估節電 kWh"], blocks["節電 %"])
]
excess_texts = [
    f"{hours} 小時<br>最多 +{max_excess:.2f}°C"
    for hours, max_excess in zip(blocks["回水超過（小時）"], blocks["回水最多超過 °C"])
]
fig.add_trace(
    go.Bar(
        x=blocks.index,
        y=blocks["推估節電 kWh"],
        marker_color=bar_colors,
        showlegend=False,
        text=saving_texts,
        textposition="outside",
        hovertemplate="%{x}｜%{y:,.0f} kWh<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=blocks.index,
        y=blocks["回水超過（小時）"],
        marker_color=bar_colors,
        showlegend=False,
        text=excess_texts,
        textposition="outside",
        hovertemplate="%{x}｜%{y} 小時<extra></extra>",
    ),
    row=2,
    col=1,
)
fig.update_yaxes(
    range=[
        min(0, blocks["推估節電 kWh"].min() * 2.6),
        blocks["推估節電 kWh"].max() * 1.35,
    ],
    row=1,
    col=1,
)
fig.update_yaxes(range=[0, blocks["回水超過（小時）"].max() * 1.45], row=2, col=1)
show(
    fig,
    "一天分幾段控制：節電與回水超過",
    f"比較時段 {N_HR} 小時＋開停機分段 {N_SEG} 段；灰色為逐時（對照）；"
    f"台數與出水都依段內第 {control.LOAD_PERCENTILE:.0f} 百分位",
    height=750,
)
blocks.set_axis(blocks.index.str.replace("<br>", ""), axis=0).round(1)

,推估節電 kWh,節電 %,回水超過（小時）,回水最多超過 °C,建議並聯（評估單位）,其中實際只開 CH-2,並聯單位節電 kWh,僅 CH-2 單位節電 kWh
控制方式,,,,,,,,
逐時,717.1,3.5,52,0.9,37,13,778.6,-61.5
3 段（每 3 小時）,543.6,2.7,39,0.7,42,17,767.8,-224.2
2 段（上午 4 小時、下午 5 小時）,433.5,2.1,34,0.5,43,19,758.7,-325.2
1 段（整天 9 小時）,271.8,1.3,30,0.5,48,23,755.6,-483.7


In [12]:
# 逐時建議下，回水超過回水上限的原因：超過量 = 流量差 + 負荷預測誤差 + 設定取整
total_rt = result["S3 CH-1 RT"] + result["S3 CH-2 RT"]
total_flow = result["S3 CH-1 LPM"] + result["S3 CH-2 LPM"]
unit_plan = recommendation["建議方案"].reindex(units.HOUR).to_numpy()
unit_prediction = hourly.LOAD_PRED.reindex(units.HOUR).to_numpy()
# 規則用的流量典型值
typical_flow = np.where(unit_plan == rules.BOTH, FLOWS["parallel"], FLOWS["single"])
excess_parts = pd.DataFrame(
    {
        "流量差（實際流量 ≠ 典型值）": total_rt / (rules.K * total_flow)
        - total_rt / (rules.K * typical_flow),
        "負荷預測誤差（實際 ≠ 預測）": (total_rt - unit_prediction)
        / (rules.K * typical_flow),
        "設定取整到 0.1°C": result["S3 CH-2 °C"]
        - (rules.T_RETURN - unit_prediction / (rules.K * typical_flow)),
    }
)
excess = bt.return_excess(result)
over = excess > 0.05
main_cause = (
    excess_parts[over]
    .idxmax(axis=1)
    .value_counts()
    .reindex(excess_parts.columns, fill_value=0)
)
at_lower_limit = (result.loc[over, "S3 CH-2 °C"] <= rules.T_MIN + 0.01).sum()
print(
    f"回水超過 0.05°C 以上 {int(over.sum())} 個評估單位"
    f"（共 {int(excess.notna().sum())} 個）："
    f"平均 +{excess[over].mean():.2f}°C、最多 +{excess.max():.2f}°C；"
    f"設定落在出水下限 {int(at_lower_limit)} 個"
)
for cause_name in excess_parts.columns:
    print(
        f"  {cause_name}：平均貢獻 {excess_parts.loc[over, cause_name].mean():+.2f}°C，"
        f"是主因的單位 {main_cause[cause_name]} 個"
    )

回水超過 0.05°C 以上 52 個評估單位（共 102 個）：平均 +0.27°C、最多 +0.86°C；設定落在出水下限 0 個
  流量差（實際流量 ≠ 典型值）：平均貢獻 +0.19°C，是主因的單位 39 個
  負荷預測誤差（實際 ≠ 預測）：平均貢獻 +0.07°C，是主因的單位 10 個
  設定取整到 0.1°C：平均貢獻 +0.01°C，是主因的單位 3 個


## 7. 控制依據百分位：節電與供冷風險的取捨

分段控制時，每段僅設定一次運轉台數與出水溫度，須以段內預測負荷的某一統計值作為控制依據。下圖比較以第 50（中位數）、75（現行）、100（最大值）百分位為依據時，各分段方式的：
- **推估節電量**（kWh）：越高越佳
- **供冷風險**（h）：越低越佳；照建議運轉時可能供冷不足的小時數，包括：
  - **回水溫度超限**：回水高於上限 13.9°C 達 0.5°C 以上，表示供水溫度偏高，末端供冷與除濕能力可能不足
  - **運轉台數不足**：建議單機運轉，但實際負荷高於 380 RT，CH-2 無法維持出水溫度

In [13]:
# 各段數在不同百分位下的節電與供冷風險（比較時段）
# 供冷風險：回水超過回水上限 0.5°C 以上、或建議只開 CH-2 但實際負荷超過並聯門檻，任一種發生的小時數
PCTS = [50, 60, 70, 75, 80, 90, 100]
compare_index = hourly.index[comparable]
percentile_rows = []
for percentile in PCTS:
    for segment_count in [3, 2, 1]:
        segment_recommendation = control.recommend(
            hourly.LOAD_PRED, FLOWS, segment_count, percentile
        )
        segment_result = bt.evaluate(
            units, segment_recommendation, curves, ch2_model, sensitivity, TYP
        )
        saving_kwh = sum(
            bt.saving(segment_result, units, step, previous)
            for step, previous in [("S1", "S0"), ("S2", "S1"), ("S3", "S2")]
        )
        hourly_excess = bt.return_excess(segment_result).groupby(units.HOUR).max()
        # 建議只開 CH-2、但實際負荷超過並聯門檻
        single_over_limit = (
            segment_recommendation.loc[compare_index, "建議方案"] == rules.CH2_ONLY
        ) & (hourly.loc[compare_index, "Q_ACT"] > rules.PARALLEL_ABOVE)
        excess_hours = set(hourly_excess.index[hourly_excess > 0.5])
        single_hours = set(compare_index[single_over_limit])
        percentile_rows.append(
            {
                "百分位": percentile,
                "段數": segment_count,
                "推估節電 kWh": saving_kwh.sum(),
                "回水超過 0.5°C 以上（小時）": len(excess_hours),
                "建議單機但實際 > 380 RT（小時）": len(single_hours),
                "供冷風險（小時）": len(excess_hours | single_hours),
            }
        )
percentile_table = pd.DataFrame(percentile_rows)

# 取捨圖：橫軸供冷風險、縱軸節電，越左上越好；只畫第 50、75（現行，大點）、100 百分位，其他看表
PLOT_PCTS = [50, 75, 100]
# 百分位數字的位置，避免和其他點的數字重疊
LABEL_POSITIONS = {50: "middle right", 75: "top left", 100: "middle left"}
COLORS = {3: "#a33f17", 2: "#eb6834", 1: "#f5a582"}
fig = go.Figure()
for segment_count, color in COLORS.items():
    is_plotted = percentile_table["百分位"].isin(PLOT_PCTS)
    segment_data = percentile_table[is_plotted & (percentile_table["段數"] == segment_count)]
    segment_name = control.SEGMENT_NAMES[segment_count]
    marker_sizes = []
    label_positions = []
    for percentile in segment_data["百分位"]:
        label_positions.append(LABEL_POSITIONS[percentile])
        if percentile == control.LOAD_PERCENTILE:
            marker_sizes.append(16)
        else:
            marker_sizes.append(9)
    fig.add_trace(
        go.Scatter(
            x=segment_data["供冷風險（小時）"],
            y=segment_data["推估節電 kWh"],
            mode="lines+markers+text",
            name=segment_name,
            text=[f"{percentile}" for percentile in segment_data["百分位"]],
            textposition=label_positions,
            textfont=dict(color=color, size=13),
            line=dict(color=color, width=2.5),
            marker=dict(size=marker_sizes, color=color),
            hovertemplate=(
                f"{segment_name}｜第 %{{text}} 百分位<br>"
                "供冷風險 %{x} 小時｜節電 %{y:,.0f} kWh<extra></extra>"
            ),
        )
    )
fig.update_xaxes(title_text="供冷風險（小時）", range=[0, 13], dtick=2)
fig.update_yaxes(title_text="推估節電 (kWh)", rangemode="tozero")
show(
    fig,
    "不同百分位的節電與供冷風險",
    "點旁數字為百分位：50＝段內中間、75＝現行（大點）、100＝段內最高；越左上越好",
    legend="bottom",
    height=600,
)
percentile_table.pivot(index="百分位", columns="段數").round(0)

推估節電 kWh               回水超過 0.5°C 以上（小時）       建議單機但實際 > 380 RT（小時）        \
段數         1      2      3                 1  2  3                    1  2  3   
百分位                                                                             
50     304.0  578.0  665.0                 2  7  4                    0  4  2   
60     294.0  449.0  651.0                 1  5  4                    0  1  2   
70     282.0  439.0  547.0                 1  1  2                    0  1  1   
75     272.0  433.0  544.0                 1  1  2                    0  1  1   
80     270.0  301.0  538.0                 1  1  2                    0  0  1   
90     251.0  291.0  530.0                 1  1  1                    0  0  1   
100    245.0  277.0  520.0                 1  1  1                    0  0  1   

    供冷風險（小時）         
段數         1   2  3  
百分位                  
50         2  11  6  
60         1   6  6  
70         1   2  3  
75         1   2  3  
80         1   1  3  
90         1   1  2  
100        1   1  2

- 第 50：最省電，但風險最高（最多 11 小時）
- 第 75：風險降到 1~3 小時，節電仍保留大部分 → 現行採用
- 第 100：風險只再少 0~1 小時，節電明顯變少
- 1 段風險都很低：實際超過 380 RT 的日子整天負荷都高，整天並聯，但節電最少
- 尖峰規則：回測期間觸發的段，第 75 百分位都已超過 380 RT（原本就並聯）；改依最高小時後出水較低，節電略減、供冷風險略降

**控制依據計算方式**
- **P75**：段內各小時預測負荷的第 75 百分位數（線性內插），約 75% 的小時負荷不高於此值；可涵蓋大部分時段，又不因單一尖峰時段而過度保守
- **尖峰規則**：段內超過 380 RT 的部分加總 > 380 × 25% = 95 RT 時，改以段內最大負荷為依據（兩台並聯），不受最短運轉 3 小時限制（例：兩小時 430、440 RT，50 + 60 = 110 > 95 → 依 440 RT）

In [14]:
# 說明圖：1 段（整天 08~16 時當一段）怎麼決定控制依據；數字為示意
HOURS = [f"{hour:02d} 時" for hour in control.WORK_HOURS]
EXAMPLES = {
    "一般情況": [300, 320, 330, 340, 350, 340, 330, 320, 300],
    "尖峰情況": [280, 300, 320, 430, 440, 340, 320, 300, 280],
}
BAR_COLOR = "#86b6ef"  # 各小時預測負荷（淺藍）
BASIS_COLOR = "#184f95"  # 第 75 百分位（深藍）
PEAK_COLOR = "#a33f17"  # 並聯門檻、超出部分與改用的依據
EXCESS_COLOR = "#f5a582"  # 超過並聯門檻的部分（淺橘）
# 橫線從第一根長條左緣畫到最後一根右緣，數字標在右側空白處
LINE_X = [-0.5, len(HOURS) - 0.5]
LABEL_X = len(HOURS) - 0.4


# 畫一天的例子：各小時預測負荷長條（超過並聯門檻的部分另標）＋第 75 百分位、並聯門檻、尖峰時改用的依據
def draw_basis_example(case_name: str, loads: list) -> None:
    loads = pd.Series(loads, dtype=float)
    percentile_load = loads.quantile(control.LOAD_PERCENTILE / 100)
    below = loads.clip(upper=rules.PARALLEL_ABOVE)
    excess = (loads - rules.PARALLEL_ABOVE).clip(lower=0)
    basis, triggered = control.segment_basis(loads)

    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=list(range(len(HOURS))),
            y=below,
            name="各小時預測負荷（示意）",
            marker_color=BAR_COLOR,
            text=[f"{load:.0f}" for load in loads],
            textposition="inside",
            insidetextanchor="start",
            customdata=loads,
            hovertemplate="%{customdata:.0f} RT<extra></extra>",
        )
    )
    fig.add_trace(
        go.Bar(
            x=list(range(len(HOURS))),
            y=excess,
            name=f"超過 {rules.PARALLEL_ABOVE} RT 的部分",
            marker_color=EXCESS_COLOR,
            showlegend=bool(excess.sum() > 0),
            text=[f"+{value:.0f}" if value > 0 else "" for value in excess],
            textposition="inside",
            textfont=dict(color=PEAK_COLOR),
            hovertemplate="超出 %{y:.0f} RT<extra></extra>",
        )
    )

    lines = [
        (percentile_load, f"第 {control.LOAD_PERCENTILE:.0f} 百分位", BASIS_COLOR, "solid"),
        (rules.PARALLEL_ABOVE, "並聯門檻", PEAK_COLOR, "dot"),
    ]
    if triggered:
        lines.append((basis, "尖峰時改用的依據（最高小時）", PEAK_COLOR, "solid"))
    for value, line_name, color, dash in lines:
        fig.add_trace(
            go.Scatter(
                x=LINE_X + [LABEL_X],
                y=[value, value, value],
                mode="lines+text",
                name=line_name,
                line=dict(color=color, width=2.5, dash=dash),
                text=["", "", f"{value:.0f}"],
                textposition="middle right",
                textfont=dict(color=color),
                hovertemplate=f"{line_name}：{value:.0f} RT<extra></extra>",
            )
        )

    fig.update_layout(barmode="stack", legend_traceorder="normal")
    fig.update_xaxes(
        tickvals=list(range(len(HOURS))),
        ticktext=HOURS,
        range=[-0.6, len(HOURS) - 0.1],
    )
    fig.update_yaxes(title_text="預測負荷 (RT)", range=[0, 500])

    # 副標：這一天的控制依據怎麼來
    excess_total = excess.sum()
    if triggered:
        subtitle = (
            f"超過 {rules.PARALLEL_ABOVE} RT 的部分合計 {excess_total:.0f} RT"
            f" > {control.PEAK_EXCESS:.0f} RT（{rules.PARALLEL_ABOVE} × {control.PEAK_EXCESS_RATIO:.0%}），"
            f"整天改依 {basis:.0f} RT 決定、兩台並聯"
        )
    else:
        subtitle = (
            f"超過 {rules.PARALLEL_ABOVE} RT 的部分合計 {excess_total:.0f} RT，"
            f"未超過 {control.PEAK_EXCESS:.0f} RT，整天依第 {control.LOAD_PERCENTILE:.0f} 百分位 {basis:.0f} RT 決定"
        )
    show(fig, f"{case_name}：1 段（整天 08~16 時）的控制依據", subtitle, legend="bottom", height=500)


for case_name, loads in EXAMPLES.items():
    draw_basis_example(case_name, loads)

## 8. 結論與限制

**結論**
- 比較時段（09/21~10/12，上班時段開機 117 小時）：建議與實際台數一致 104 小時；不一致的 13 小時中，12 小時是建議並聯、實際只開 CH-2
- 逐時照建議操作，推估節電 **717 kWh（3.5%）**：並聯出水（兩台同一設定）+1,028 kWh 最多，單機出水 +218 kWh；台數 −528 kWh，因為預測超過 380 RT 就建議並聯，以供冷穩定為優先
- 業主可選的分段控制：3 段 544 kWh（2.7%）、2 段 433 kWh（2.1%）、1 段 272 kWh（1.3%）；段數越少，越常整段並聯，節電越少
- 控制依據百分位的合理範圍為第 70~75（現行 75）

**限制**
- 回測期間約 4 週；預測負荷為訓練資料內召回，上線時誤差會較大
- 實際回水約一半時段略高於回水上限 13.9°C（平均 +0.27°C），主因是實際流量低於流量典型值
- CH-2 建議出水多數超出灰箱模型範圍（8.5~9.8°C），該部分節電為外推；CH-1 出水影響沿用 CH-2 敏感度估計
- 建議並聯但實際只開 CH-2 的時段，CH-1 出水和流量用並聯時中位數推估，需試行確認
- 未計冷卻水塔、冰水泵耗電；規則不使用天氣資料